[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-svm.ipynb)

# Support Vector Machines

*AIBits Academy · Machine Learning End To End · Maximum Margin Classifier*

SVMs find the hyperplane that maximises the margin between classes — a geometrically elegant solution with powerful kernel extensions for non-linear data.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## The Maximal Margin Hyperplane

Given linearly separable data, infinitely many hyperplanes separate the classes. SVM selects the one that maximises the **margin** — the distance between the hyperplane and the nearest points of each class (the **support vectors**).

$$\begin{gathered}\text{Maximise:}\ \dfrac{2}{\lVert w\rVert} \quad \text{Subject to:}\ y_i(\mathbf{w}\cdot x_i+b)\ge 1\ \ \forall i\\[8pt]\text{Equivalently: Minimise}\ \tfrac12\lVert w\rVert^2 \quad \text{s.t. } y_i(\mathbf{w}\cdot x_i+b)\ge 1\end{gathered}$$

> **📊 Prerequisite refresher**
>
> The ‖w‖ here is exactly the L2 norm from the **Linear Algebra for ML** prerequisite page's Vector Norms section — the same norm that shows up as the Ridge penalty in Regularization. Maximising the margin 2/‖w‖ is equivalent to minimising ‖w‖ itself, which is exactly why the objective flips to minimising ½‖w‖² instead.

## Soft Margin — Handling Non-Separable Data

Real data is rarely perfectly separable. The soft-margin SVM introduces slack variables ξᵢ ≥ 0 allowing some misclassifications:

$$\text{Minimise } \tfrac12\lVert w\rVert^2 + C\sum \xi_i \quad \text{s.t. } y_i(\mathbf{w}\cdot x_i+b)\ge 1-\xi_i$$

**C** controls the trade-off: large C = penalise misclassifications heavily (small margin, possible overfit); small C = allow more violations (large margin, better generalisation).

> **💡 Why the Widest Margin, Not Just Any Separator**
>
> Infinitely many hyperplanes can separate two classes perfectly on the training data — SVM insists on the one farthest from both. Picture a delivery rider threading between two rows of parked autos in a crowded Bengaluru market lane: hug one row too closely and a single auto door opening mid-ride causes a collision; stay in the dead centre of the widest gap and there's room to absorb small surprises on either side. The maximum-margin hyperplane is exactly this — the separator most robust to small perturbations in new, unseen points, which is why it tends to generalise better than any of the other equally-correct-on-training-data alternatives.

## SVM Margin Visualisation

A real soft-margin linear SVM fit on Jaipur textile QC data (fabric strength vs thread count → pass/fail). Every slider position shows an *actual sklearn solution* — the solid line is the separating hyperplane **w**·x + b = 0, the dashed lines are the margins **w**·x + b = ±1, and the yellow rings mark the true support vectors at that C. Drag C and watch the classic trade-off: small C tolerates margin violations and spreads the margin wide across the gap; large C insists on respecting the two closest "sentry" points, narrowing the margin and rotating the boundary.

## The Dual Problem — Why Kernels Become Possible

The primal problem above optimises directly over w and b. Using Lagrange multipliers αᵢ ≥ 0 for each constraint, the primal can be transformed into an equivalent **dual** problem:

$$\text{Maximise:}\quad \sum_i \alpha_i - \tfrac12\sum_i\sum_j \alpha_i\alpha_j y_i y_j (x_i\cdot x_j) \quad \text{subject to: } \alpha_i\ge 0,\ \sum_i \alpha_i y_i = 0$$

The critical observation: the dual objective depends on the training data **only through dot products** xᵢ·xⱼ — never on the raw feature vectors individually. This is precisely what enables the kernel trick: replace every dot product xᵢ·xⱼ with a kernel function K(xᵢ,xⱼ) that computes the dot product *as if* the data had been mapped into a much higher-dimensional space φ(x), without ever explicitly computing φ(x). At the optimum, most αᵢ are exactly zero — only the support vectors (points on or inside the margin) have αᵢ > 0, which is why SVM predictions depend only on a small subset of training points.

## Kernel Trick — Non-Linear SVM

When data is not linearly separable, the **kernel trick** implicitly maps features to a higher-dimensional space where linear separation is possible — without computing the mapping explicitly.

| Kernel | Formula K(a,b) | Use for |
|---|---|---|
| Linear | aᵀb | Linearly separable, high-dim text |
| RBF / Gaussian | exp(−γ‖a−b‖²) | Non-linear, most common default |
| Polynomial | (γaᵀb + r)^d | Image classification |
| Sigmoid | tanh(γaᵀb + r) | Neural-network-like problems |

## With scikit-learn — RBF Kernel

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
import numpy as np

# Jaipur textile QC: multi-class (Grade A/B/C) from fabric parameters
np.random.seed(42)
n=450
strength   = np.random.normal(150,30,n)
thread_cnt = np.random.normal(100,20,n)
shrinkage  = np.random.normal(5,1.5,n)
X=np.column_stack([strength, thread_cnt, shrinkage])
y=(((strength-150)/30 + (thread_cnt-100)/20) // 1.5).clip(-1,1).astype(int)+1

X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,random_state=42)
sc=StandardScaler(); X_tr_s=sc.fit_transform(X_tr); X_te_s=sc.transform(X_te)

# Grid search over C and gamma
param_grid={'C':[0.1,1,10], 'gamma':['scale',0.1,1]}
gs=GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='accuracy')
gs.fit(X_tr_s, y_tr)
print(f"Best params: {gs.best_params_}")
print(f"CV accuracy: {gs.best_score_:.3f}")
print(classification_report(y_te, gs.predict(X_te_s),
      target_names=['Grade C','Grade B','Grade A']))

> **🔗 Real-World Link — Click-Through Rate Prediction**
>
> An RBF-kernel SVM on 10,000 real ad impressions reaches 72.0% accuracy — nearly identical to a plain Logistic Regression baseline, a genuine lesson in when the fancier kernel isn't worth it. [See the case study →](https://statso.io/click-through-rate-analysis-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · A linear SVM and its support vectors

Fit `SVC(kernel="linear", C=1.0)` on the two well-separated groups. Store the predicted class of the point `(5, 5)` in `label` and the number of support vectors in `n_sv`.

In [ ]:
import numpy as np
from sklearn.svm import SVC
X = np.array([[1, 1], [2, 1], [1, 2], [2, 2], [8, 8], [9, 8], [8, 9], [9, 9]])
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])
label = n_sv = None   # TODO


In [ ]:
try:
    check("(5,5) is on the midline; a linear SVM returns a class", label in (0, 1))
    check("only a few support vectors", 2 <= n_sv <= 4)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.svm import SVC
X = np.array([[1, 1], [2, 1], [1, 2], [2, 2], [8, 8], [9, 8], [8, 9], [9, 9]])
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])
svm = SVC(kernel="linear", C=1.0).fit(X, y)
label = int(svm.predict([[5, 5]])[0])
n_sv = int(svm.n_support_.sum())

```

</details>

### Exercise 2 · Medium · Kernel trick versus a straight line

The rings below cannot be split by a line. Compare test accuracy of a linear and an RBF `SVC`; store them in `acc_linear`, `acc_rbf` and set `rbf_wins`.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
X, y = make_circles(n_samples=400, noise=0.08, factor=0.4, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
acc_linear = acc_rbf = rbf_wins = None   # TODO


In [ ]:
try:
    check("linear is near chance", acc_linear < 0.7)
    check("RBF is accurate", acc_rbf > 0.95)
    check("flag", rbf_wins is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import make_circles
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
X, y = make_circles(n_samples=400, noise=0.08, factor=0.4, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
acc_linear = SVC(kernel="linear").fit(X_tr, y_tr).score(X_te, y_te)
acc_rbf = SVC(kernel="rbf").fit(X_tr, y_tr).score(X_te, y_te)
rbf_wins = bool(acc_rbf > acc_linear)

```

</details>

### Exercise 3 · Stretch · Grid-search C and gamma

Run `GridSearchCV` over `C` in [0.1, 1, 10] and `gamma` in [0.1, 1, 10] (RBF kernel, 5-fold). Store the fitted search in `grid`; `best_params` = `grid.best_params_`.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
X, y = make_moons(n_samples=300, noise=0.25, random_state=1)
grid = best_params = None   # TODO


In [ ]:
try:
    check("nine candidates tried", len(grid.cv_results_["params"]) == 9)
    check("params reported", set(best_params) == {"C", "gamma"})
    check("good CV accuracy", grid.best_score_ > 0.85)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import make_moons
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
X, y = make_moons(n_samples=300, noise=0.25, random_state=1)
grid = GridSearchCV(SVC(kernel="rbf"), {"C": [0.1, 1, 10], "gamma": [0.1, 1, 10]}, cv=5).fit(X, y)
best_params = grid.best_params_

```

Large gamma makes each point's influence tiny (wiggly boundary); large C punishes every training error (also wiggly). Search them together.

</details>

---
*Back to the course: **Machine Learning End To End → Support Vector Machines**.*